# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import duckdb
import pandas as pd
from google.colab import userdata

# 1. Retrieve Hugging Face token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    print("✅ Found HF_TOKEN in secrets.")
except Exception as e:
    raise ValueError("❌ Please store your Hugging Face read token in Colab Secrets named 'HF_TOKEN'.")

# 2. Connect DuckDB & setup Hugging Face authentication
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Base path for FlyRank Hugging Face Parquet dataset
BASE_DATA_PATH = "hf://datasets/FlyRank/internship-warehouse"

# Quick connectivity test on daily performance table
row_count = con.sql(f"SELECT COUNT(*) FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/**/*.parquet')").fetchone()[0]
print(f"✅ Successfully connected to FlyRank Warehouse dataset! Total rows in fact table: {row_count:,}")

✅ Found HF_TOKEN in secrets.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Successfully connected to FlyRank Warehouse dataset! Total rows in fact table: 78,835,655


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis & Time Window Statement

* **One row =** One unique combination of `client_hash_id` + `content_hash_id` + `report_date` (in `fact_content_daily_performance`).
* **Over which dates =** The mid-panel observation month from **March 1, 2026 (`2026-03-01`) to March 31, 2026 (`2026-03-31`)** (`month=2026-03`).

#### Empirical Verification Results:
* **Date Range:** 2026-03-01 to 2026-03-31
* **Total Rows:** 9,841,378
* **Unique Grain Count:** 9,841,378
* **Uniqueness Status:** PASSED (Zero duplicate rows across client + content + date).

In [5]:
# =====================================================================
# SECTION 1 VERIFICATION: GRAIN AND DATE RANGE IN WAREHOUSE (FIXED SCHEMA)
# =====================================================================

# First, let's inspect the actual column names in the parquet files
schema_df = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

# Detect the date column dynamically (usually 'date' or 'report_date')
date_col = [c for c in schema_df['column_name'].tolist() if 'date' in c.lower()]
date_col_name = date_col[0] if date_col else 'date'

verification_query = f"""
SELECT
    MIN({date_col_name}) AS start_date,
    MAX({date_col_name}) AS end_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST({date_col_name} AS VARCHAR))) AS unique_grain_count
FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

res = con.sql(verification_query).df()
display(res)

# Print explicit proof statement
is_unique = res['total_rows'][0] == res['unique_grain_count'][0]
print(f"\n✅ Date Range: From {res['start_date'][0]} to {res['end_date'][0]}")
print(f"✅ Total Rows: {res['total_rows'][0]:,}")
print(f"✅ Grain Uniqueness Check: {'PASSED (One row = unique client + content + date)' if is_unique else 'FAILED'}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows,unique_grain_count
0,2026-03-01,2026-03-31,9841378,9841378



✅ Date Range: From 2026-03-01 00:00:00 to 2026-03-31 00:00:00
✅ Total Rows: 9,841,378
✅ Grain Uniqueness Check: PASSED (One row = unique client + content + date)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: Feature / Label / Context / Excluded

Every field in the dataset that is accessed or evaluated is categorized into one of four functional buckets below:

---

### 1. 🎯 Label Bucket (Target & Proxy Metrics)
Fields used to construct ground-truth targets or calculate business optimization metrics:
* **`gsc_clicks` / `gsc_impressions`**: Historical performance metrics used to derive actual observed Click-Through Rate ($\text{CTR} = \frac{\text{clicks}}{\text{impressions}}$).
* **`expected_ctr`**: Continuous regression target predicted by the ML model based on features.
* **`ctr_gap`**: Derived business proxy label computed via rule: $\text{Predicted Expected CTR} - \text{Actual Observed CTR}$.

---

### 2. ⚡ Feature Bucket (Model Predictors)
Input fields used by the model to predict expected engagement and CTR potential:
* **`gsc_position`**: Search engine result page (SERP) average position (strong non-linear relationship with CTR).
* **`ga4_total_engagement_sec`**: On-page user engagement duration.
* **`ai_gemini` / `ai_meta` / `ai_other`**: Indicators/scores for AI-assisted content optimization or generation.
* **`gsc_data_available` / `ga4_data_available`**: Boolean flags indicating data completeness for tracking integration.

---

### 3. 🏷️ Context Bucket (Identifiers & Partitioning)
Primary keys, foreign keys, and temporal boundaries used to structure the grain and aggregate data:
* **`client_hash_id`**: Unique identifier for the client account.
* **`content_hash_id`**: Unique identifier for the specific content/URL piece.
* **`report_date`**: Daily temporal timestamp defining the primary observation grain.
* **`month`**: Partition key (e.g., `month=2026-03`) used for temporal scoping and directory filtering.

---

### 4. 🚫 Excluded Bucket (With Justifications)
Fields omitted from model training to prevent data leakage, redundancy, or poor signal quality:

| Field Name | Reason for Exclusion |
| :--- | :--- |
| **`client_has_gsc` / `client_has_ga4`** | **Redundant with row-level availability flags:** High-level client toggles duplicate the row-level `gsc_data_available` and `ga4_data_available` signals without adding predictive value. |
| **`June 2026 Data (`month=2026-06`)`** | **Data Leakage / Sealed Outcome Window:** June 2026 represents the final future test month. Excluded from feature engineering and training pipelines to avoid lookahead bias. |
| **Raw Unaggregated Text Identifiers** | **High Cardinality / Non-Predictive:** Raw hash strings (`client_hash_id`, `content_hash_id`) serve purely as joins/groupings in Context, not direct input features for statistical prediction. |

In [6]:
# =====================================================================
# SECTION 2: FIELD BUCKET CLASSIFICATION & DICTIONARY REGISTRATION
# =====================================================================

field_buckets = {
    "label": [
        "gsc_clicks",
        "gsc_impressions",
        "expected_ctr",
        "ctr_gap"
    ],
    "feature": [
        "gsc_position",
        "ga4_total_engagement_sec",
        "ai_gemini",
        "ai_meta",
        "ai_other",
        "gsc_data_available",
        "ga4_data_available"
    ],
    "context": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "month"
    ],
    "excluded": {
        "client_has_gsc": "Redundant with row-level gsc_data_available flag",
        "client_has_ga4": "Redundant with row-level ga4_data_available flag",
        "month=2026-06": "Sealed future test month; excluded to prevent data leakage"
    }
}

# Verify bucket counts
print("✅ Field Buckets successfully registered:")
for category, fields in field_buckets.items():
    if isinstance(fields, list):
        print(f"  • {category.upper():<10} ({len(fields)} fields): {', '.join(fields)}")
    else:
        print(f"  • {category.upper():<10} ({len(fields)} rules/fields):")
        for f, reason in fields.items():
            print(f"      - {f}: {reason}")

✅ Field Buckets successfully registered:
  • LABEL      (4 fields): gsc_clicks, gsc_impressions, expected_ctr, ctr_gap
  • FEATURE    (7 fields): gsc_position, ga4_total_engagement_sec, ai_gemini, ai_meta, ai_other, gsc_data_available, ga4_data_available
  • CONTEXT    (4 fields): client_hash_id, content_hash_id, report_date, month
  • EXCLUDED   (3 rules/fields):
      - client_has_gsc: Redundant with row-level gsc_data_available flag
      - client_has_ga4: Redundant with row-level ga4_data_available flag
      - month=2026-06: Sealed future test month; excluded to prevent data leakage


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1: Grain Uniqueness Check
**Claim:** One row in our slice represents exactly one unique combination of `client_hash_id` + `content_hash_id` + `report_date`.

In [7]:
# =====================================================================
# QUERY 1: VERIFY GRAIN UNIQUENESS
# =====================================================================

q1_grain_check = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST(report_date AS VARCHAR))) AS unique_grain_count,
    COUNT(*) - COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST(report_date AS VARCHAR))) AS duplicate_count
FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

res_q1 = con.sql(q1_grain_check).df()
display(res_q1)

print(f"✅ Claim Verified: Total rows ({res_q1['total_rows'][0]:,}) matches unique grain count ({res_q1['unique_grain_count'][0]:,}). Zero duplicates.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_count,duplicate_count
0,9841378,9841378,0


✅ Claim Verified: Total rows (9,841,378) matches unique grain count (9,841,378). Zero duplicates.


### Query 2: Slice Row Count and Date Span
**Claim:** The mid-panel slice for `month=2026-03` spans precisely from `2026-03-01` to `2026-03-31` with complete daily coverage.

In [8]:
# =====================================================================
# QUERY 2: VERIFY DATE SPAN AND ROW COUNT
# =====================================================================

q2_span_check = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_days,
    COUNT(*) AS total_slice_rows
FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

res_q2 = con.sql(q2_span_check).df()
display(res_q2)

print(f"✅ Claim Verified: Date span covers {res_q2['distinct_days'][0]} days from {res_q2['min_date'][0]} to {res_q2['max_date'][0]} across {res_q2['total_slice_rows'][0]:,} rows.")

,min_date,max_date,distinct_days,total_slice_rows
0,2026-03-01,2026-03-31,31,9841378


✅ Claim Verified: Date span covers 31 days from 2026-03-01 00:00:00 to 2026-03-31 00:00:00 across 9,841,378 rows.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.